In [ ]:
import pandas as pd 
import numpy as np

Creating a function to clean and load the dataset 

In [ ]:
def wrangle(filepath):
    
    #Reading the data set 
    df = pd.read_csv(filepath, compression = 'gzip')
    
    for column in df.columns:
        
        #Removing the irrelevant columns 
        if df[column].dtype == 'object':
            df.drop(columns = [column], inplace = True)
        else:
            #cleansing the data set for nan, infinity and large data type
            if np.any(np.isnan(df[column])) or np.any(np.isinf(df[column])) or np.any(np.abs(df[column]) > np.finfo(np.float64).max):

                #Replace it with median values of that columns
                median_value = np.median(df[column][np.isfinite(df[column])])
                df[column].replace([np.nan, np.inf, -np.inf], median_value, inplace = True)

    return df

In [ ]:
df = wrangle('C:/Users/akwar/Downloads/data_assignment.gz')
print(df.shape)
df.head()

# Split

In [ ]:
#Creating Feature matrix 
target = 'ns'
y = df[target]
X = df.drop(columns = target)

print("X shape:", X.shape)
print("Y shape:", y.shape)

Diving the data set into training and testing 

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Resampling the data with over sampling due to imbalance dataset 

In [ ]:
from imblearn.over_sampling import RandomOverSampler

over_sampler = RandomOverSampler(random_state = 42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print("X_train_over shape:", X_train_over.shape)
X_train_over.head()

As we can see from the above that the size of the dataset has increased due to resampling

# Model Building

In [ ]:
acc_baseline = y_train.value_counts(normalize = True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))

Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier()

tree.fit(X_train_over, y_train_over)
print(f"Decision tree has {tree.tree_.node_count} nodes with maximum depth {tree.tree_.max_depth}.")

In [ ]:
#Probability Prediction 
train_probs = tree.predict_proba(X_train_over)[:, -1]
probs = tree.predict_proba(X_test)[:, -1]

train_predictions = tree.predict(X_train_over)
predictions = tree.predict(X_test)

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, roc_curve

print(f" Train ROC AUC Score: {roc_auc_score(y_train_over, train_probs)}")
print(f"Test ROC AUC Score: {roc_auc_score(y_test, probs)}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix


In [ ]:
ConfusionMatrixDisplay.from_estimator(tree, X_test, y_test);

In [ ]:
class_report = classification_report(y_test, tree.predict(X_test))
print(class_report)

Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    
    SimpleImputer(), RandomForestClassifier(random_state = 42)
)

Performing cross validation with classifier using over_sampled training data. 

In [ ]:
cv_scores = cross_val_score(clf, X_train_over, y_train_over, cv = 5, n_jobs =1)
print(cv_scores)

In [ ]:
#Creating a dictionary for range for hyper parameters. 
params = {
    "simpleimputer__strategy": ['mean', 'median'],
    "randomforestclassifier__n_estimators": range(25,100,25),
    "randomforestclassifier__max_depth": range(10,50,10)
}

Creating a GridSeachCV that includes classifier and hyperparameter grid.

In [ ]:
model = GridSearchCV(
    clf, param_grid = params, cv =5, n_jobs =-1, verbose = 1
)

In [ ]:
model.fit(X_train_over, y_train_over)

In [ ]:
cv_results = pd.DataFrame(model.cv_results_)
cv_results.head(5)

In [ ]:
#Extracting the best hyperparameters
model.best_params_

In [ ]:
acc_train = model.score(X_train, y_train)
acc_test = model.score(X_test, y_test)

print("Model Training Accuracy:", round(acc_train, 4))
print("Model Test Accuracy:", round(acc_test, 4))

In [ ]:
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test);

In [ ]:
class_report = classification_report(y_test, model.predict(X_test))
print(class_report)

In [ ]:
#Get important features
features = X_train_over.columns 

#Extract importances from model
importances = model.best_estimator_.named_steps['randomforestclassifier'].feature_importances_

#Create a series with feature names and importances 
feat_imp = pd.Series(importances, index=features).sort_values()

In [ ]:
import matplotlib.pyplot as plt

#Plot the important features
feat_imp.tail(20).plot(kind = "barh")
plt.xlabel("Gini Importance")
plt.ylabel("Features")
plt.title("Feature Importance")

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def make_cnf_matrix(threshold):
    
    y_pred_proba = model.predict_proba(X_test)[:, -1]
    y_pred = y_pred_proba >= threshold
    conf_matrix = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = conf_matrix.ravel()
    
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, colorbar = False)

In [ ]:
thresh_widget = widgets.FloatSlider(min = 0, max = 1, value = 0.2, step = 0.2)
interact(make_cnf_matrix, threshold = thresh_widget);

In [ ]:
#Calculating for best threshold value 
best_threshold = None 
max_expected_value = float('-inf')

for threshold in np.linspace(0, 1, 100): # Using 100 thresholds between 0 and 1
    
    y_pred_proba = model.predict_proba(X_test)[:, -1]
    y_pred = y_pred_proba >= threshold
    
    #Compute confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = conf_matrix.ravel()
    
    #Define benefit and cost values
    B_TP = 400
    B_TN = 0
    C_FP = -200
    C_FN = -150
    
    #Define the benefit and cost values
    EV = TP* B_TP + TN * B_TN + FP * C_FP + FN * C_FN
    
    #Check if this threshold gives a higher expected values
    if EV > max_expected_value:
        max_expected_value = EV 
        best_threshold = threshold
        
print("Best Threshold:", best_threshold)
print("Max Expected Value:", max_expected_value)